# えんじいろ Google Colab 機械学習環境

このノートブックは、google/gemma-3-1b-it を4bitで読み込み、QLoRAの最小スモーク学習からLoRAアダプター再読込、推論、localhost限定API確認までを上から順に検証するためのものです。

- Google Colabの一時ランタイムを前提とし、本番環境ではありません。
- GPUは保証されません。GPUがない場合は、重い処理をCPUへ切り替えず明示的に停止します。
- 既定値は SMOKE_TEST=True、FULL_TRAIN=False。学習は最大2 stepです。
- 秘密情報・学習済み重み・ベースモデルをGitへ保存しません。
- モデルの役割はテキスト変換だけです。モデレーションや投稿可否判定は対象外です。

## 1. 設定

まず既定値を確認します。長時間学習は明示的に FULL_TRAIN=True へ変更しない限り実行されません。

In [ ]:
from pathlib import Path
from typing import Literal
import os

MODEL_ID = "google/gemma-3-1b-it"
SMOKE_TEST = True
FULL_TRAIN = False

MAX_SEQ_LENGTH = 256
MAX_NEW_TOKENS = 96
MAX_OUTPUT_CHARS = 150

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
NUM_TRAIN_EPOCHS = 2
SMOKE_MAX_STEPS = 2
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.0

USE_GOOGLE_DRIVE = False
SAVE_ADAPTER = True
DATA_PATH = None
REPOSITORY_URL = "https://github.com/engiiro/engiiro-app.git"
WORK_ROOT = Path("/content/engiiro-work")
REPO_DIR = WORK_ROOT / "engiiro-app"
ADAPTER_DIR = WORK_ROOT / "artifacts" / "gemma-3-1b-it-engiiro-lora"

assert not (SMOKE_TEST and FULL_TRAIN), "SMOKE_TESTとFULL_TRAINは同時に有効化しません"
print({"SMOKE_TEST": SMOKE_TEST, "FULL_TRAIN": FULL_TRAIN, "MODEL_ID": MODEL_ID})

## 2. ランタイム情報

CPU・GPU・VRAM・CUDA・BF16対応・RAM・主要ライブラリの実測値を表示します。GPUがなければ、以降のモデル読込や学習は実行しません。

In [ ]:
import importlib.metadata as md
import platform
import subprocess
import sys

import torch

def package_version(name: str) -> str:
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return "not installed"

gpu_available = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if gpu_available else None
gpu_vram_gib = (
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
    if gpu_available else None
)
try:
    import psutil
    ram_gib = round(psutil.virtual_memory().total / 1024**3, 2)
except ImportError:
    ram_gib = "psutil unavailable"
runtime_report = {
    "python": sys.version.split()[0],
    "os": platform.platform(),
    "cpu": platform.processor() or "unknown",
    "ram_gib": ram_gib,
    "gpu_available": gpu_available,
    "gpu_name": gpu_name,
    "gpu_vram_gib": gpu_vram_gib,
    "cuda": torch.version.cuda,
    "bf16_supported": bool(gpu_available and torch.cuda.is_bf16_supported()),
    "packages": {
        name: package_version(name)
        for name in ("torch", "scikit-learn", "fastapi", "uvicorn", "pydantic", "transformers", "peft", "bitsandbytes", "accelerate", "datasets")
    },
}
runtime_report

## 3. privateリポジトリの取得

Colabの「シークレット」へ GITHUB_TOKEN を登録してください。トークンはURLへ埋め込まず、一時的な GIT_ASKPASS 経由でのみ渡します。出力にも表示しません。既存cloneがあればremote URLを検査します。

In [ ]:
import stat
import tempfile

def require_colab_secret(name: str) -> str:
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except ImportError as exc:
        raise RuntimeError("このセルはGoogle Colabで実行してください") from exc
    except Exception as exc:
        raise RuntimeError(f"Colabシークレット {name} を設定してください") from exc
    if not value:
        raise RuntimeError(f"Colabシークレット {name} が空です")
    return str(value)

WORK_ROOT.mkdir(parents=True, exist_ok=True)
github_token = require_colab_secret("GITHUB_TOKEN")
askpass_path = Path(tempfile.gettempdir()) / "engiiro_git_askpass.py"
askpass_path.write_text(
    "#!/usr/bin/env python3\n"
    "import os, sys\n"
    "print('x-access-token' if 'Username' in sys.argv[1] else os.environ['ENGIIRO_GITHUB_TOKEN'])\n",
    encoding="utf-8",
)
askpass_path.chmod(askpass_path.stat().st_mode | stat.S_IEXEC)
git_env = {
    **os.environ,
    "GIT_ASKPASS": str(askpass_path),
    "GIT_TERMINAL_PROMPT": "0",
    "ENGIIRO_GITHUB_TOKEN": github_token,
}
try:
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], env=git_env, check=True)
    else:
        subprocess.run(["git", "clone", "--quiet", REPOSITORY_URL, str(REPO_DIR)], env=git_env, check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPOSITORY_URL], check=True)
    remote_output = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "remote", "-v"], text=True
    )
    assert github_token not in remote_output
    assert REPOSITORY_URL in remote_output
    print(remote_output)
finally:
    os.environ.pop("ENGIIRO_GITHUB_TOKEN", None)
    if askpass_path.exists():
        askpass_path.unlink()
    git_env.clear()
    github_token = None

## 4. Hugging Face認証とGemmaアクセス確認

Colabの「シークレット」へ HF_TOKEN を登録し、Hugging Face上でGemmaのライセンス同意とアクセス許可を済ませてください。401（認証失敗）と403（アクセス拒否）は区別して停止し、別モデルへフォールバックしません。

In [ ]:
import urllib.error
import urllib.request
import json

hf_token = require_colab_secret("HF_TOKEN")

def hf_json(url: str) -> dict:
    request = urllib.request.Request(
        url,
        headers={"Authorization": f"Bearer {hf_token}", "User-Agent": "engiiro-colab-environment-check"},
    )
    try:
        with urllib.request.urlopen(request, timeout=30) as response:
            return json.load(response)
    except urllib.error.HTTPError as exc:
        if exc.code == 401:
            raise RuntimeError("HF_TOKENの認証に失敗しました（401）") from exc
        if exc.code == 403:
            raise PermissionError(f"{MODEL_ID}へのアクセスが拒否されました（403）。ライセンス同意を確認してください") from exc
        raise

whoami = hf_json("https://huggingface.co/api/whoami-v2")
model_metadata = hf_json(f"https://huggingface.co/api/models/{MODEL_ID}")
print({"hf_user": whoami.get("name"), "model_id": model_metadata.get("id"), "access_check": "metadata OK"})

## 5. 依存関係

Colab既定の PyTorch と scikit-learn は入れ替えません。QLoRAに必要なパッケージだけを最低バージョンで確認し、不足または古いものだけを導入します。実際に使われるバージョンを再表示します。

In [ ]:
from importlib import invalidate_caches
from packaging.version import Version

requirements = {
    "transformers": ("4.51.0", "transformers>=4.51,<5"),
    "peft": ("0.15.0", "peft>=0.15,<1"),
    "bitsandbytes": ("0.45.0", "bitsandbytes>=0.45,<1"),
    "accelerate": ("1.4.0", "accelerate>=1.4,<2"),
    "datasets": ("3.3.0", "datasets>=3.3,<5"),
    "fastapi": ("0.115.0", "fastapi>=0.115,<1"),
    "uvicorn": ("0.30.0", "uvicorn>=0.30,<1"),
    "pydantic": ("2.9.0", "pydantic>=2.9,<3"),
}
for core_package in ("torch", "scikit-learn"):
    if package_version(core_package) == "not installed":
        raise RuntimeError(
            f"Colab既定の{core_package}がありません。自動再インストールはせず、ランタイム情報を報告してください。"
        )

to_install = []
for package, (minimum, specifier) in requirements.items():
    current = package_version(package)
    if current == "not installed" or Version(current) < Version(minimum):
        to_install.append(specifier)

if to_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *to_install])
    invalidate_caches()

resolved_versions = {
    name: package_version(name)
    for name in ("torch", "scikit-learn", *requirements.keys())
}
print({"installed_now": to_install, "resolved_versions": resolved_versions})

### 5.1 CPU環境スモークテスト

GPUの有無にかかわらず、既存のCPU環境要件を先に確認します。小さなTensor演算、固定文字列のTF-IDF・fit・predict、既存 ai/app.py のimportと /health 応答だけが対象です。モデル品質、API契約、変換ロジックは評価しません。

In [ ]:
import importlib.util

cpu_tensor = torch.tensor([[1.0, 2.0], [3.0, 4.0]], device="cpu")
cpu_tensor_result = (cpu_tensor @ cpu_tensor).tolist()
assert cpu_tensor.device.type == "cpu"
assert cpu_tensor_result == [[7.0, 10.0], [15.0, 22.0]]

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

fixed_texts = ["赤ちゃんの言葉です", "やさしい母の言葉です", "幼い話し方です", "丁寧な母の表現です"]
fixed_labels = ["baby", "mother", "baby", "mother"]
vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(1, 2))
features = vectorizer.fit_transform(fixed_texts)
classifier = LogisticRegression(random_state=0).fit(features, fixed_labels)
prediction = classifier.predict(vectorizer.transform(["丁寧でやさしい母の言葉です"]))
assert prediction.shape == (1,)

ai_dir = REPO_DIR / "ai"
app_path = ai_dir / "app.py"
if not app_path.is_file():
    raise FileNotFoundError(f"既存AIアプリがありません: {app_path}")
sys.path.insert(0, str(ai_dir))
try:
    spec = importlib.util.spec_from_file_location("engiiro_colab_app", app_path)
    app_module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(app_module)
finally:
    sys.path.remove(str(ai_dir))

health_route = next(
    route for route in app_module.app.routes
    if getattr(route, "path", None) == "/health" and "GET" in getattr(route, "methods", set())
)
health_result = health_route.endpoint()
assert health_result == {"status": "ok"}
print({
    "torch_cpu_tensor": "PASS",
    "sklearn_fit_predict": "PASS",
    "ai_app_import": "PASS",
    "health": health_result,
    "non_goals": ["model quality", "API contract", "evaluate/transform correctness"],
})

## 6. 学習データの読込と検証

JSONLまたはCSVを受け付け、各行に mode、input、output の3列を必須とします。mode は baby または mother です。Google DriveやColabアップロードも選べます。既定の4件は秘密情報を含まない環境確認専用サンプルで、品質評価用データではありません。生データ全体は表示しません。

In [ ]:
import csv
from collections import Counter
from io import BytesIO, StringIO

DEFAULT_SAMPLES = [
    {"mode": "baby", "input": "今日はレビューでたくさん学びました。", "output": "きょうは れびゅーで いっぱい おべんきょうしたよ！"},
    {"mode": "baby", "input": "エラーの原因を調査しています。", "output": "えらーさんが どこにいるか さがしてるの。"},
    {"mode": "mother", "input": "今日はレビューでたくさん学びました。", "output": "今日はレビューを通して、多くの学びを得ました。"},
    {"mode": "mother", "input": "エラーの原因を調査しています。", "output": "現在、エラーの原因を丁寧に確認しています。"},
]

def records_from_path(path: Path) -> list[dict]:
    if path.suffix.lower() == ".jsonl":
        with path.open(encoding="utf-8") as stream:
            return [json.loads(line) for line in stream if line.strip()]
    if path.suffix.lower() == ".csv":
        with path.open(encoding="utf-8-sig", newline="") as stream:
            return list(csv.DictReader(stream))
    raise ValueError("DATA_PATHは.jsonlまたは.csvにしてください")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

if DATA_PATH == "UPLOAD":
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("アップロードするデータファイルは1つにしてください")
    uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
    upload_path = WORK_ROOT / Path(uploaded_name).name
    upload_path.write_bytes(uploaded_bytes)
    records = records_from_path(upload_path)
elif DATA_PATH:
    records = records_from_path(Path(DATA_PATH))
else:
    records = list(DEFAULT_SAMPLES)

required_fields = {"mode", "input", "output"}
normalized_records = []
for index, row in enumerate(records, start=1):
    if not isinstance(row, dict) or not required_fields.issubset(row):
        raise ValueError(f"{index}行目にmode/input/outputがありません")
    clean = {key: str(row[key]).strip() for key in required_fields}
    if clean["mode"] not in {"baby", "mother"}:
        raise ValueError(f"{index}行目のmodeはbabyまたはmotherにしてください")
    if not clean["input"] or not clean["output"]:
        raise ValueError(f"{index}行目に空文字があります")
    normalized_records.append(clean)

pairs = [(row["mode"], row["input"], row["output"]) for row in normalized_records]
duplicate_count = len(pairs) - len(set(pairs))
if duplicate_count:
    raise ValueError(f"完全重複データが{duplicate_count}件あります")

data_report = {
    "source": "built-in smoke samples" if not DATA_PATH else str(DATA_PATH),
    "record_count": len(normalized_records),
    "mode_counts": dict(Counter(row["mode"] for row in normalized_records)),
    "duplicates": duplicate_count,
    "purpose": "environment smoke test only",
}
data_report

## 7. Gemma 3 1Bの4bit読込

GPUがない場合はここで停止します。NF4、double quant、BF16対応GPUではBF16・それ以外ではFP16を使います。OOM時は設定を自動変更して再試行せず、計測値とともに停止します。Gemmaへアクセスできない場合も別モデルへ切り替えません。

In [ ]:
if not gpu_available:
    raise RuntimeError(
        "GPUを取得できませんでした。ColabのランタイムをGPUへ変更して最初から再実行してください。"
        "モデル読込・学習・推論・API実測は未検証です。"
    )

from huggingface_hub.errors import GatedRepoError
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
memory_before_gib = round(torch.cuda.memory_allocated() / 1024**3, 3)
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        token=hf_token,
        quantization_config=quantization_config,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
except torch.cuda.OutOfMemoryError as exc:
    peak = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
    raise RuntimeError(
        f"Gemmaの4bit読込でCUDA OOMが発生しました。peak_vram_gib={peak}。"
        "自動再試行や設定変更は行っていません。"
    ) from exc
except GatedRepoError as exc:
    raise PermissionError(
        f"{MODEL_ID}のgated accessがありません。ライセンス同意とHF_TOKENを確認してください。"
        "別モデルへのフォールバックは行いません。"
    ) from exc

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
memory_after_gib = round(torch.cuda.memory_allocated() / 1024**3, 3)
peak_load_gib = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
print({
    "model_id": MODEL_ID,
    "compute_dtype": str(compute_dtype),
    "memory_before_gib": memory_before_gib,
    "memory_after_gib": memory_after_gib,
    "peak_load_gib": peak_load_gib,
})

## 8. QLoRA設定と学習対象の確認

実モデルに存在する4bit線形層から対象モジュール名を求め、q/k/v/o投影とMLP投影だけへLoRAを付けます。学習可能パラメータにLoRA以外、埋め込み、lm_headが混ざっていないことをassertで確認します。

In [ ]:
import bitsandbytes as bnb
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

candidate_leaf_names = {
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
}
linear4bit_leaf_names = {
    name.rsplit(".", 1)[-1]
    for name, module in model.named_modules()
    if isinstance(module, bnb.nn.Linear4bit)
}
target_modules = sorted(candidate_leaf_names & linear4bit_leaf_names)
if not target_modules:
    raise RuntimeError(f"LoRA対象層を実モデルから検出できませんでした: {sorted(linear4bit_leaf_names)}")

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.gradient_checkpointing_enable()
model.config.use_cache = False
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=target_modules,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

trainable_names = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
assert trainable_names, "学習可能パラメータがありません"
assert all("lora_" in name for name in trainable_names), trainable_names
assert not any("lm_head" in name or "embed_tokens" in name for name in trainable_names)
trainable_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
total_count = sum(parameter.numel() for parameter in model.parameters())
print({
    "target_modules": target_modules,
    "trainable_parameters": trainable_count,
    "total_parameters": total_count,
    "trainable_percent": round(trainable_count / total_count * 100, 4),
})

## 9. 最小スモーク学習

既存のGemma chat templateを使い、特殊トークンを手作業で足しません。既定では最大2 stepだけ実行し、checkpointを保存しません。FULL_TRAINを明示した場合だけ最大2 epoch、epochごと・最大2個のcheckpointを許可します。

In [ ]:
import math
import time

from datasets import Dataset
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

MODE_INSTRUCTIONS = {
    "baby": "入力文の意味を保ちながら、幼い子どものような柔らかい表現へ変換してください。",
    "mother": "入力文の意味を保ちながら、見守る母親のような丁寧で温かい表現へ変換してください。",
}

def format_training_record(row: dict) -> str:
    messages = [
        {
            "role": "user",
            "content": f"{MODE_INSTRUCTIONS[row['mode']]}\n入力: {row['input']}\n150文字以内で出力してください。",
        },
        {"role": "assistant", "content": row["output"]},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

formatted_dataset = Dataset.from_list(
    [{"text": format_training_record(row)} for row in normalized_records]
)

def tokenize_record(row: dict) -> dict:
    encoded = tokenizer(
        row["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
        add_special_tokens=False,
    )
    encoded["labels"] = list(encoded["input_ids"])
    return encoded

tokenized_dataset = formatted_dataset.map(
    tokenize_record,
    remove_columns=formatted_dataset.column_names,
)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_enabled = SMOKE_TEST or FULL_TRAIN
training_args = TrainingArguments(
    output_dir=str(WORK_ROOT / "trainer-output"),
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    max_steps=SMOKE_MAX_STEPS if SMOKE_TEST else -1,
    num_train_epochs=NUM_TRAIN_EPOCHS if FULL_TRAIN else 1,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    optim="paged_adamw_8bit",
    bf16=compute_dtype == torch.bfloat16,
    fp16=compute_dtype == torch.float16,
    logging_steps=1,
    save_strategy="no" if SMOKE_TEST else ("epoch" if FULL_TRAIN else "no"),
    save_total_limit=2 if FULL_TRAIN else None,
    report_to=[],
    remove_unused_columns=False,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

training_report = {"executed": False, "reason": "both flags are false"}
if training_enabled:
    torch.cuda.reset_peak_memory_stats()
    started_at = time.perf_counter()
    train_result = trainer.train()
    elapsed_seconds = round(time.perf_counter() - started_at, 2)
    final_loss = float(train_result.training_loss)
    if not math.isfinite(final_loss):
        raise RuntimeError(f"学習lossが有限値ではありません: {final_loss}")
    training_report = {
        "executed": True,
        "mode": "smoke" if SMOKE_TEST else "full",
        "steps": int(trainer.state.global_step),
        "loss": final_loss,
        "elapsed_seconds": elapsed_seconds,
        "peak_vram_gib": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }
    if SMOKE_TEST:
        assert trainer.state.global_step <= 2
training_report

## 10. LoRAアダプターだけを保存・再読込

保存先はリポジトリ外です。USE_GOOGLE_DRIVE=True の場合だけGoogle Driveへ保存します。フルモデルのファイルがないことを確認し、mergeせずにLoRAを外した同じベースモデルへアダプターを再読込します。

In [ ]:
from peft import PeftModel

adapter_output_dir = (
    Path("/content/drive/MyDrive/engiiro/artifacts/gemma-3-1b-it-lora")
    if USE_GOOGLE_DRIVE
    else ADAPTER_DIR
)
adapter_report = {"saved": False}
if SAVE_ADAPTER:
    adapter_output_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(adapter_output_dir, safe_serialization=True)
    tokenizer.save_pretrained(adapter_output_dir)

    saved_names = {path.name for path in adapter_output_dir.iterdir() if path.is_file()}
    assert "adapter_config.json" in saved_names
    assert "adapter_model.safetensors" in saved_names
    forbidden_full_model_files = {"model.safetensors", "pytorch_model.bin"}
    assert not (saved_names & forbidden_full_model_files), saved_names

    base_model = model.unload()
    del model
    model = PeftModel.from_pretrained(base_model, adapter_output_dir, is_trainable=False)
    del base_model
    adapter_size_bytes = sum(
        path.stat().st_size for path in adapter_output_dir.rglob("*") if path.is_file()
    )
    adapter_report = {
        "saved": True,
        "path": str(adapter_output_dir),
        "files": sorted(saved_names),
        "size_mib": round(adapter_size_bytes / 1024**2, 2),
        "merged": False,
        "reloaded": True,
    }
adapter_report

## 11. 推論と150文字制約

同じ入力を baby / mother の両モードで確認します。推論は同時実行数1、入力最大256 token、生成最大96 token、batch size 1です。150文字を超えた場合は一度だけ短文化を再依頼し、それでも超えたら切り捨てずエラーにします。

In [ ]:
from threading import Lock

inference_lock = Lock()

class OutputTooLongError(RuntimeError):
    pass

def _generate_once(mode: Literal["baby", "mother"], text: str, retry: bool = False) -> str:
    if mode not in MODE_INSTRUCTIONS:
        raise ValueError("modeはbabyまたはmotherにしてください")
    if not isinstance(text, str) or not text.strip():
        raise ValueError("textは空でない文字列にしてください")

    constraint = (
        "前回の出力は150文字を超えました。意味を保ったまま必ず150文字以内に短くしてください。"
        if retry
        else "必ず150文字以内で出力してください。"
    )
    messages = [{
        "role": "user",
        "content": f"{MODE_INSTRUCTIONS[mode]}\n{constraint}\n入力: {text.strip()}",
    }]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        add_special_tokens=False,
    ).to(model.device)
    with torch.inference_mode():
        generated = model.generate(
            **encoded,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = generated[0, encoded["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def transform_text(mode: Literal["baby", "mother"], text: str) -> str:
    with inference_lock:
        result = _generate_once(mode, text)
        if len(result) <= MAX_OUTPUT_CHARS:
            return result
        result = _generate_once(mode, text, retry=True)
        if len(result) > MAX_OUTPUT_CHARS:
            raise OutputTooLongError(
                f"再生成後も{len(result)}文字でした。出力は切り捨てていません。"
            )
        return result

same_input = "今日はチームで仕様書をレビューし、未決事項を整理しました。"
inference_report = {
    mode: transform_text(mode, same_input)
    for mode in ("baby", "mother")
}
inference_report

## 12. localhost限定Uvicornヘルスチェック

既存プロセスがあれば止め、worker 1、127.0.0.1限定で一時APIを起動します。公開トンネルは作りません。モデルAPI統合ではなく、Colab内でUvicornを起動できることと /health の正確な応答だけを確認します。

In [ ]:
import time
from urllib.request import urlopen

if "api_process" in globals() and api_process.poll() is None:
    api_process.terminate()
    api_process.wait(timeout=10)

api_process = subprocess.Popen(
    [
        sys.executable, "-m", "uvicorn", "app:app",
        "--host", "127.0.0.1", "--port", "8000",
        "--workers", "1", "--log-level", "warning",
    ],
    cwd=REPO_DIR / "ai",
)

health_payload = None
deadline = time.time() + 30
while time.time() < deadline:
    if api_process.poll() is not None:
        raise RuntimeError(f"Uvicornが終了しました: returncode={api_process.returncode}")
    try:
        with urlopen("http://127.0.0.1:8000/health", timeout=2) as response:
            health_payload = json.load(response)
        break
    except Exception:
        time.sleep(1)

assert health_payload == {"status": "ok"}, health_payload
print({
    "application": str(REPO_DIR / "ai" / "app.py"),
    "host": "127.0.0.1",
    "port": 8000,
    "workers": 1,
    "public_tunnel": False,
    "health": health_payload,
})

## 13. Uvicorn停止

確認後はこのセルを実行して一時サーバーを停止します。

In [ ]:
if "api_process" in globals() and api_process.poll() is None:
    api_process.terminate()
    try:
        api_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        api_process.kill()
        api_process.wait(timeout=5)
print("Uvicorn stopped")

## 14. 後片付け

一時askpass、トークン変数、Uvicorn、モデル参照、CUDAキャッシュを片付けます。remote URLに資格情報が残っていないことも再確認します。Google Driveは自動unmountしません。

In [ ]:
import gc
import re

if "api_process" in globals() and api_process.poll() is None:
    api_process.terminate()
    try:
        api_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        api_process.kill()

if "askpass_path" in globals() and askpass_path.exists():
    askpass_path.unlink()
if "git_env" in globals():
    git_env.clear()
for secret_name in (
    "ENGIIRO_GITHUB_TOKEN",
    "GITHUB_TOKEN",
    "HF_TOKEN",
    "HUGGING_FACE_HUB_TOKEN",
    "HUGGINGFACE_HUB_TOKEN",
):
    os.environ.pop(secret_name, None)
github_token = None
hf_token = None

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPOSITORY_URL], check=True)
    remote_output = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "remote", "-v"], text=True
    )
    assert not re.search(r"https://[^/\s]+@github\.com/", remote_output)
    print(remote_output)

for object_name in ("trainer", "model", "tokenizer"):
    if object_name in globals():
        del globals()[object_name]
gc.collect()
torch.cuda.empty_cache()
print({
    "secrets_cleared": True,
    "temporary_askpass_removed": not askpass_path.exists(),
    "google_drive_unmounted": False,
})

## 次の手順

- 本ノートブックの既定実行は最大2 stepの環境確認です。長時間のFULL_TRAINは今回実行しません。
- 実データを使う前に、mode/input/outputの定義、利用許諾、個人情報除去、評価基準を人間が確認してください。
- 保存対象はLoRAアダプターとtokenizer設定だけです。ベースモデルや秘密情報をGitへ追加しないでください。
- Gemmaはテキスト変換だけを担当します。モデレーション、投稿可否、認証、永続化、公開API化は別責務です。
- Colabランタイムは停止すると消えるため、継続利用するアダプターだけを明示的にGoogle Drive等へ退避してください。